<a href="https://colab.research.google.com/github/MZiaAfzal71/Melting-Point-Prediction-of-Boronic-Acids/blob/main/Data%20Files/Scripts%20and%20Models/Generate_fingerprints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Notebook Title: Descriptor Generation for Boronic Acids
## 🧾 Overview

This notebook consolidates the full molecular descriptor generation pipeline for boronic acid compounds, using the cleaned dataset obtained from Organoborons.com.
It generates multiple feature sets — including MACCS, Morgan, Mordred, Coulomb Matrix, and the custom — Our descriptor — and saves each in a separate Excel file.
All descriptors are later used for property prediction models.

# ⚙️ **0. Environment Setup (Important Step!)**

Before running the notebook, please install the required scientific chemistry libraries — **RDKit**, **Mordred**, and **DScribe** — which are essential for generating molecular descriptors.  

📦 These libraries are not preinstalled in Google Colab.  
After installation, **you must restart the runtime** to ensure all dependencies load correctly before executing the next cells.


In [ ]:
!pip install rdkit mordred dscribe

## 🧩 1. Clone Repository and Navigate to Working Directory

The following cell clones the GitHub repository “Melting-Point-Prediction-of-Boronic-Acids” and changes the current working directory to the folder containing the data files, scripts, and pre-trained models used in this project.

In [1]:
!git clone https://github.com/MZiaAfzal71/Melting-Point-Prediction-of-Boronic-Acids
%cd Melting-Point-Prediction-of-Boronic-Acids/Data\ Files/Scripts\ and\ Models

Cloning into 'Melting-Point-Prediction-of-Boronic-Acids'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 118 (delta 22), reused 3 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 24.09 MiB | 14.10 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/Melting-Point-Prediction-of-Boronic-Acids/Data Files/Scripts and Models


## 🧩 2. Imports and Setup

In [2]:
# Core imports
import pandas as pd
import numpy as np
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit.Chem.rdMolDescriptors import GetMACCSKeysFingerprint
from mordred import Calculator, descriptors
import networkx as nx
from dscribe.descriptors import CoulombMatrix
import ase
from tqdm.auto import tqdm

## 🧱 3. Utility Functions

In [3]:
def smiles_str_to_rdkit_mol(smiles_str: str) -> rdkit.Chem.Mol:
    """
    Convert a SMILES string to an RDKit mol object and generate its 3D structure.

    Args:
        smiles_str (str): A SMILES string representing a molecule.

    Returns:
        rdkit.Chem.Mol: An RDKit mol object representing the molecule with an optimized 3D structure.
    """
    try:
        # Convert SMILES string to RDKit mol object
        mol = Chem.MolFromSmiles(smiles_str)
        if mol is None:
            raise ValueError("Invalid SMILES string")

        # Add hydrogens to the molecule
        mol = Chem.AddHs(mol)

        # Generate 3D coordinates using ETKDG
        if AllChem.EmbedMolecule(mol, AllChem.ETKDG()) != 0:
            raise ValueError("Failed to embed molecule")

        # Optimize 3D geometry using UFF force field
        AllChem.UFFOptimizeMolecule(mol)

        return mol
    except Exception as e:
        print(f"Error processing SMILES '{smiles_str}': {e}")
        return None


def ase_atoms_to_coulomb_matrix(ase_atoms: ase.Atoms) -> np.ndarray:
    """
    Convert an ASE Atoms object to a Coulomb matrix.

    Args:
        ase_atoms (ase.Atoms): ASE Atoms object.

    Returns:
        np.ndarray: Flattened Coulomb matrix representation of the molecule.
    """
    try:
        # Create a Coulomb matrix descriptor
        coulomb_matrix = CoulombMatrix(n_atoms_max=ase_atoms.get_global_number_of_atoms())

        # Compute the Coulomb matrix
        col_matrix = coulomb_matrix.create(ase_atoms)

        # Reshape the matrix for easier processing
        return col_matrix.flatten()
    except Exception as e:
        print(f"Error generating Coulomb matrix: {e}")
        return None

def get_mol_graph(mol):
    """Construct a graph from an RDKit molecule (with explicit hydrogens).
    Nodes: atom indices.
    Edges: bonds with attributes: bond_type and aromatic flag."""
    G = nx.Graph()
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        bt = bond.GetBondType()
        aromatic = bond.GetIsAromatic()
        G.add_edge(i, j, bond_type=bt, aromatic=aromatic)
    return G


def compute_path_sum(mol, path, G):
    """
    Compute the path sum for a given path.
    For each atom along the path, its atomic number (Z) is added.
    For the bond connecting the previous atom, apply a multiplier:
      - Aromatic bond: 1.5
      - Double bond (non-aromatic): 2.0
      - Triple bond (non-aromatic): 3.0
      - Otherwise: 1.0
    The reference atom (first in the path) is added normally.
    """
    total = 0.0
    for idx in range(len(path)):
        atom = mol.GetAtomWithIdx(path[idx])
        Z = atom.GetAtomicNum()
        if idx == 0:
            total += Z
        else:
            data = G.get_edge_data(path[idx - 1], path[idx])
            multiplier = 1.0
            if data.get("aromatic"):
                multiplier = 1.5
            else:
                bond_type = data.get("bond_type")
                if bond_type == Chem.rdchem.BondType.DOUBLE:
                    multiplier = 2.0
                elif bond_type == Chem.rdchem.BondType.TRIPLE:
                    multiplier = 3.0
            total += multiplier * Z
    return total


def compute_descriptor_vector(mol):
    """
    Compute a descriptor vector from the molecule.
    The molecule is first converted to include explicit hydrogens.
    The longest bond path length from the Boron reference atom is used as max_depth.
    For each atom at a bond distance d (d>=2) from the Boron reference,
    all shortest paths are obtained and the path sum is computed.
    The descriptor vector component for a given d is the cumulative sum over all such paths.
    """
    # Convert to include explicit hydrogens
    mol = Chem.AddHs(mol)

    # Identify Boron as the reference atom (choose the first Boron found)
    boron_indices = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetSymbol() == 'B']
    if not boron_indices:
        return None
    ref = boron_indices[0]

    # Build graph
    G = get_mol_graph(mol)

    # Compute the shortest path lengths from reference
    lengths = nx.single_source_shortest_path_length(G, ref)
    if not lengths:
        return None
    max_depth = max(lengths.values())

    descriptor = {}  # keys: bond distance; values: cumulative path sum
    for atom_idx, dist in lengths.items():
        # Only consider atoms with distance >= 2
        if dist < 2:
            continue
        # Get all shortest paths between ref and this atom
        paths = list(nx.all_shortest_paths(G, source=ref, target=atom_idx))
        path_sum_total = sum(compute_path_sum(mol, path, G) for path in paths)
        descriptor.setdefault(dist, 0.0)
        descriptor[dist] += path_sum_total

    # Build the descriptor vector for distances 2 to max_depth
    vector = [descriptor.get(d, 0.0) for d in range(2, max_depth + 1)]
    return np.array(vector)

## 🔹 4. Generate MACCS and Morgan Fingerprints

In [ ]:
# Input / Output paths
smiles_file = "Excel Files/Cleaned_Boronic_Acids.xlsx"
output_morgan = "Excel Files/Boronic_Morgan_fingerprint.xlsx"
output_maccs = "Excel Files/Boronic_MACCS_fingerprint.xlsx"

df = pd.read_excel(smiles_file)

descMorgan_list, descMACCS_list = [], []

mol_gen = GetMorganGenerator(radius=2, fpSize=512)

for i, smiles in tqdm(enumerate(df['SMILES']), total=len(df)):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue
    try:
        mol_3d = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol_3d, AllChem.ETKDG())
        AllChem.UFFOptimizeMolecule(mol_3d)

        descMorgan_list.append(list(mol_gen.GetFingerprint(mol)))
        descMACCS_list.append(list(GetMACCSKeysFingerprint(mol)))
    except:
        continue

results_morgan = pd.DataFrame(descMorgan_list, columns=[f"desc_{i}" for i in range(512)])
results_maccs = pd.DataFrame(descMACCS_list, columns=[f"desc_{i}" for i in range(len(descMACCS_list[0]))])

base_cols = ['Name', 'SMILES', 'Melting Point']
final_morgan = pd.concat([df[base_cols], results_morgan], axis=1)
final_maccs = pd.concat([df[base_cols], results_maccs], axis=1)

final_morgan.to_excel(output_morgan, index=False)
final_maccs.to_excel(output_maccs, index=False)
print("✅ MACCS & Morgan fingerprints saved.")


## ⚛️ 5. Generate Coulomb Matrix Descriptors

In [ ]:
# Define file paths (keeping them unchanged)
input_file = 'Excel Files/Cleaned_Boronic_Acids.xlsx'
output_file = 'Excel Files/CoulombMatrix_BoronicAcids_Desc.xlsx'

# Load input Excel file
chem_file = pd.read_excel(input_file)

# Initialize list for storing descriptors
descriptors_list = []

# Process each molecule in the dataset
for i, sm in tqdm(enumerate(chem_file['SMILES']), total=len(chem_file)):
    try:
        mol = smiles_str_to_rdkit_mol(sm)
        if mol:
            # Convert RDKit mol to ASE Atoms object
            ase_atoms = ase.Atoms(
                numbers=[atom.GetAtomicNum() for atom in mol.GetAtoms()],
                positions=mol.GetConformer().GetPositions()
            )

            # Compute the Coulomb matrix
            col_mat = ase_atoms_to_coulomb_matrix(ase_atoms)

            # Append the descriptor to the list
            if col_mat is not None:
                descriptors_list.append(col_mat)
            else:
                descriptors_list.append([None] * 100)  # Placeholder for missing data

        else:
            descriptors_list.append([None] * 100)  # Placeholder for missing data

    except Exception as e:
        descriptors_list.append([None] * 100)


# Convert descriptors list into a DataFrame
desc_df = pd.DataFrame(descriptors_list)

# Assign meaningful column names
desc_df.columns = [f"desc_{i}" for i in range(desc_df.shape[1])]

# Extract essential base columns
base_cols = ['Name', 'SMILES', 'Melting Point']
base_df = chem_file[base_cols].copy()

# Concatenate base data with descriptors
final_df = pd.concat([base_df, desc_df], axis=1)

# Save the final dataset to Excel
final_df.to_excel(output_file, index=False)

print("✅ Coulomb Matrix descriptors saved.")


## 🧮 6. Generate Mordred 3D Descriptors

In [ ]:
smiles_file = "Excel Files/Cleaned_Boronic_Acids.xlsx"
df = pd.read_excel(smiles_file)

# Initialize Mordred Calculator for 2D and 3D descriptors
calc_3d = Calculator(descriptors)  # Compute both 2D & 3D descriptors

# Store results
descriptors3d_list = []

for index, smiles in tqdm(enumerate(df['SMILES']), total=len(df)):
    mol = Chem.MolFromSmiles(Chem.MolToSmiles(Chem.MolFromSmiles(smiles)))  # Convert SMILES to RDKit Mol
    if mol is None:
        print(f"Invalid SMILES at row {index}: {smiles}")
        continue
    try:
        # Generate 3D Conformer
        mol_3d = Chem.AddHs(mol)  # Add hydrogen atoms
        AllChem.EmbedMolecule(mol_3d, AllChem.ETKDG())  # Generate 3D structure
        AllChem.UFFOptimizeMolecule(mol_3d)  # Optimize 3D geometry

        # Compute 3D Descriptors
        descriptors_3d = calc_3d(mol_3d)

        # Store results in a dictionary
        descriptors3d_list.append({"SMILES": smiles, **dict(descriptors_3d)})
    except:
        print(f'{index} is not being processed!')

# Convert to DataFrame and save results
results_df = pd.DataFrame(descriptors3d_list)
results_df.to_excel("Excel Files/Boronic_Mordred_3D.xlsx", index=False)

print("✅ Mordred descriptors saved.")

## 🧹 7. Clean Mordred Descriptors

In [7]:
# Define file paths
input_file = 'Excel Files/Boronic_Mordred_3D.xlsx'       # File containing Mordred descriptors
input_file1 = 'Excel Files/Cleaned_Boronic_Acids.xlsx'   # File containing cleaned boronic acids data
output_file = 'Excel Files/Boronic_Mordred_3DC.xlsx'    # Output file path

# Load Excel files into DataFrames
df1 = pd.read_excel(input_file)   # Data with Mordred descriptors
df2 = pd.read_excel(input_file1)  # Cleaned Boronic Acids data

# Select only numeric columns from df1 (Mordred descriptor data)
df3 = df1.select_dtypes(include=['number'])

# Remove columns where all values are 0 (i.e., keep only meaningful descriptors)
df3 = df3.loc[:, (df3 != 0).any(axis=0)]

# Define base columns from the cleaned Boronic Acids data
base_cols = ['Name', 'SMILES', 'Melting Point']

# Extract relevant base columns from df2
new_df = df2[base_cols].copy()

# Concatenate the base columns with the descriptor columns
final_df = pd.concat([new_df, df3], axis=1)

# Save the final merged DataFrame to an Excel file
final_df.to_excel(output_file, index=False)

print("✅ Cleaned Mordred descriptors saved.")

✅ Cleaned Mordred descriptors saved.


## 🧩 8. Generate our Descriptors

In [8]:
# Define file paths
input_file = 'Excel Files/Cleaned_Boronic_Acids.xlsx'
output_file = 'Excel Files/Boronic_Bonds_Desc_Boron_En.xlsx'
df = pd.read_excel(input_file)

# 2. For each SMILES code, convert it to canonical form and compute the descriptor vector.
descriptor_list = []
for smi in df['SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        descriptor_list.append(None)
        continue
    canonical_smi = Chem.MolToSmiles(mol, canonical=True)
    mol = Chem.MolFromSmiles(canonical_smi)

    vec = compute_descriptor_vector(mol)
    descriptor_list.append(vec)

# Convert the list of descriptors (each a list of numbers) into a DataFrame.
# This will create separate columns for each element in the descriptor.
desc_df = pd.DataFrame(descriptor_list)
# Optionally, rename the descriptor columns (e.g., desc_0, desc_1, ...)
desc_df.columns = [f"desc_{i}" for i in range(desc_df.shape[1])]

# Create a new dataframe with the desired columns from the original data:
# 'Name', 'SMILES', and 'boiling_point'
base_cols = ['Name', 'SMILES', 'Melting Point']
new_df = df[base_cols].copy()

# Concatenate the base columns with the descriptor columns
final_df = pd.concat([new_df, desc_df], axis=1)

# Save the new dataframe to an Excel file
final_df.to_excel(output_file, index=False)


print("✅ Our descriptor file saved.")

✅ Our descriptor file saved.


##  🧠 9. Summary
All descriptors generated successfully:

✅ Boronic_Morgan_fingerprint.xlsx

✅ Boronic_MACCS_fingerprint.xlsx

✅ CoulombMatrix_BoronicAcids_Desc.xlsx

✅ Boronic_Mordred_3DC.xlsx

✅ Boronic_Bonds_Desc_Boron_En.xlsx